In [0]:

silver_path = "abfss://silver-layer@stockdatalake1.dfs.core.windows.net/stock"


df_gold = spark.read.format("delta").load(silver_path)


df_gold.show(5)

In [0]:
df_gold.display()

In [0]:
df_monthly_gold = (
    df_gold.groupBy("year", "month")
    .agg(
        F.avg("Close").alias("avg_close"),
        F.max("High").alias("max_high"),
        F.min("Low").alias("min_low"),
        F.sum("shares_traded").alias("total_volume"),
        F.sum("Turnover_cr").alias("total_turnover")
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_weekly_gold = (
    df_gold.withColumn("week", F.date_format("date", "yyyy-ww"))  # Week of year
    .groupBy("symbol", "week")
    .agg(
        F.avg("Close").alias("avg_close"),
        F.max("High").alias("max_high"),
        F.min("Low").alias("min_low"),
        F.sum("shares_traded").alias("total_volume"),
        F.sum("Turnover_cr").alias("total_turnover")
    )
    .orderBy("symbol", "week")
)

In [0]:
df_gold.display()

In [0]:

df_gold.write.mode("overwrite").saveAsTable("gold_catalog.gold_schema.gold_table")

In [0]:

spark.sql("SHOW TABLES").show()


spark.sql("SELECT * FROM gold_table LIMIT 10").show()